# Head Sampling, Tail Sampling and What Each One Throws Away

Companion notebook for the [Sampling Strategies](https://codemia.io/courses/observability_and_delivery/obs_sampling_strategies) lesson on Codemia.

**What you will build**: a population of two hundred thousand traces with the properties real traces have: a heavy latency
tail, errors concentrated in the slow traces, and a rare endpoint that matters more than its volume.
You will run five sampling policies over it and measure what each one keeps, what each one costs in
spans and in collector memory, and which of them quietly ruins your latency percentiles.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2024)

TRACES = 200_000
TRACE_RATE = 5_000          # traces per second in production
BYTES_PER_SPAN = 400        # serialised, compressed

## Concept recap

At any real volume you cannot keep every trace, so the question is never whether to sample. It is
when the decision is made.

Head sampling decides at the root span, before anything is known about the request, and propagates
the decision down the call tree in the trace flags. It is cheap, it needs no buffering, and it is
blind: the interesting traces are only in your sample if they were lucky.

Tail sampling buffers the spans of a trace until it is complete, then decides with the whole trace
in hand. It can keep every error and every slow request, and it pays for that with memory
proportional to your span rate, with a hard dependency on traces finishing inside the buffer window,
and with a sample that no longer represents the population.

Those last five words are the part people discover too late, so this notebook measures them.

## A population of traces worth sampling

In [ ]:
# Most requests are short. A small class of them is not: batch endpoints, long polls,
# exports. That class is a rounding error in trace count and dominates everything else.
is_long = rng.random(TRACES) < 0.004

spans = np.clip(rng.lognormal(np.log(18), 0.7, TRACES).astype(int), 1, 400)
spans[is_long] = np.clip(rng.lognormal(np.log(90), 0.6, is_long.sum()).astype(int), 1, 2_000)

latency_ms = rng.lognormal(np.log(100), 0.95, TRACES)
latency_ms[is_long] = rng.lognormal(np.log(30_000), 0.9, is_long.sum())

# Errors cluster in slow traces, as they do in every real service: timeouts, exhausted
# retries, saturated pools. A small floor of errors is fast, a validation rejection say.
p_error = 0.002 + 0.60 / (1 + np.exp(-(np.log(latency_ms) - np.log(3_000)) / 0.5))
is_error = rng.random(TRACES) < p_error

# One route matters far more than its volume: refunds. 1 trace in 5,000.
is_rare_route = rng.random(TRACES) < 0.0002

true_p50, true_p99, true_p999 = np.percentile(latency_ms, [50, 99, 99.9])
SLOW_MS = true_p99
SECONDS = TRACES / TRACE_RATE
PER_DAY = 86_400 / SECONDS

print(f'traces                 {TRACES:>12,}   ({SECONDS:.0f} seconds at '
      f'{TRACE_RATE:,} traces/s)')
print(f'spans                  {spans.sum():>12,}   median {int(np.median(spans))} per trace, '
      f'max {spans.max()}')
print(f'raw volume             {spans.sum() * BYTES_PER_SPAN / 2 ** 20:>12,.0f} MB   '
      f'{spans.sum() * BYTES_PER_SPAN * PER_DAY / 2 ** 40:,.1f} TB/day')
print()
print(f'latency p50 {true_p50:>8.0f} ms     p99 {true_p99:>8.0f} ms     '
      f'p99.9 {true_p999:>8.0f} ms')
print(f'error traces           {is_error.sum():>12,}   {is_error.mean():.2%}')
print(f'error rate in the slowest 1% of traces  {is_error[latency_ms >= SLOW_MS].mean():>7.2%}')
print(f'error rate in the fastest half          {is_error[latency_ms < true_p50].mean():>7.2%}')
print(f'refund route traces    {is_rare_route.sum():>12,}   {is_rare_route.mean():.3%}')
print(f'long running traces    {is_long.sum():>12,}   {is_long.mean():.2%} of traces, '
      f'{spans[is_long].sum() / spans.sum():.1%} of spans')

Three properties of that population do all the damage later. Errors are two orders of magnitude more
likely in the slow tail than in the fast half, so any sampler that selects on latency is also
selecting on failure. The route you most want to debug is two hundredths of a percent of traffic,
which no uniform policy will ever hand you enough of. And a few traces in a thousand run for tens of
seconds and carry a disproportionate share of all spans, which is the detail that decides whether
tail sampling is affordable.

## Five policies

In [ ]:
HEAD_RATE = 0.01
TAIL_BASE_RATE = 0.01

policies = {}
policies['head 1%'] = rng.random(TRACES) < HEAD_RATE
policies['head 10%'] = rng.random(TRACES) < 0.10
policies['head 1% + refunds always'] = policies['head 1%'] | is_rare_route
policies['tail: errors + slow + 1%'] = (
    is_error | (latency_ms >= SLOW_MS) | (rng.random(TRACES) < TAIL_BASE_RATE))
policies['tail: errors only + 0.1%'] = is_error | (rng.random(TRACES) < 0.001)


def report(name, keep):
    kept_spans = spans[keep].sum()
    row = {
        'traces': int(keep.sum()),
        'trace %': keep.mean(),
        'span %': kept_spans / spans.sum(),
        'GB/day': kept_spans * BYTES_PER_SPAN * PER_DAY / 2 ** 30,
        'errors kept': is_error[keep].sum() / max(is_error.sum(), 1),
        'refunds kept': int(is_rare_route[keep].sum()),
        'p99 estimate': np.percentile(latency_ms[keep], 99) if keep.sum() else float('nan'),
    }
    return row


print(f'{"policy":>28}{"traces":>10}{"trace %":>9}{"span %":>8}{"GB/day":>9}'
      f'{"errors kept":>13}{"refunds":>9}{"p99 seen":>11}')
rows = {}
for name, keep in policies.items():
    r = report(name, keep)
    rows[name] = r
    print(f'{name:>28}{r["traces"]:>10,}{r["trace %"]:>9.2%}{r["span %"]:>8.2%}'
          f'{r["GB/day"]:>9.0f}{r["errors kept"]:>13.1%}{r["refunds kept"]:>9}'
          f'{r["p99 estimate"]:>9.0f} ms')
print()
print(f'{"the true population":>28}{TRACES:>10,}{1:>9.2%}{1:>8.2%}'
      f'{spans.sum() * BYTES_PER_SPAN * PER_DAY / 2 ** 30:>9.0f}'
      f'{1:>13.1%}{is_rare_route.sum():>9}{true_p99:>9.0f} ms')

The last column is the one to read twice. Head sampling lands near the population p99, a little off
at one percent and close at ten, because a uniform sample is noisy but unbiased and the noise shrinks
as the rate rises. The tail policy reports a p99 roughly a hundred times too high, and no amount of
extra data would fix it, because the quantity it selects on is the quantity you are trying to
measure. Its top percentile is made entirely of the long running class, so the number it reports is
the p99 of your worst requests rather than of your traffic.

Both behaviours are correct answers to different questions. Head sampling is a miniature of
production that you can compute statistics on and that contains almost none of your failures. Tail
sampling is a library of failures with no statistics in it.

Two smaller details in the same table are worth noticing. The tail policy keeps 2.45 percent of
traces and 3.80 percent of spans, because selecting on failure also selects the largest traces, so
its bill grows faster than its trace count suggests. And head sampling at one percent found one
refund trace in forty seconds, which is the subject of the last section.

The practical consequence is the rule broken on every second dashboard: compute latency percentiles
from metrics, which see every request, and use traces to explain the shape rather than to measure
it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
bins = np.logspace(0, 4.5, 60)

axes[0].hist(latency_ms, bins=bins, weights=np.full(TRACES, 1 / TRACES),
             color='lightgrey', label='population')
axes[0].hist(latency_ms[policies['head 1%']], bins=bins,
             weights=np.full(policies['head 1%'].sum(), 1 / policies['head 1%'].sum()),
             histtype='step', color='tab:blue', label='head 1% sample')
axes[0].set_xscale('log')
axes[0].set_title('Head sampling keeps the shape')
axes[0].set_xlabel('latency (ms)')
axes[0].set_ylabel('share of the sample')
axes[0].legend()

tail = policies['tail: errors + slow + 1%']
axes[1].hist(latency_ms, bins=bins, weights=np.full(TRACES, 1 / TRACES),
             color='lightgrey', label='population')
axes[1].hist(latency_ms[tail], bins=bins,
             weights=np.full(tail.sum(), 1 / tail.sum()),
             histtype='step', color='tab:red', label='tail sample')
axes[1].set_xscale('log')
axes[1].set_title('Tail sampling keeps the interesting part')
axes[1].set_xlabel('latency (ms)')
axes[1].legend()
plt.show()

The right hand panel is not a worse sample, it is a different object: a catalogue of incidents rather
than a survey of traffic. Treat it as a catalogue and it is the most useful data you own. Treat it as
a survey and it will tell you your service is a hundred times slower than it is.

## What tail sampling costs before it decides anything

In [ ]:
def buffered_spans(window_s, trace_rate=TRACE_RATE):
    """Every in-flight trace is held in memory until its decision window closes."""
    return trace_rate * window_s * spans.mean()


# A trace's wall clock duration is a little longer than its root span latency.
duration_ms = latency_ms * rng.uniform(1.0, 1.6, TRACES)

print(f'{"window":>9}{"spans buffered":>17}{"collector memory":>19}'
      f'{"traces still open":>20}{"their share of spans":>23}')
for window in [1, 5, 10, 30, 60, 120]:
    held = buffered_spans(window)
    open_traces = duration_ms > window * 1000
    print(f'{window:>7} s{held:>17,.0f}'
          f'{held * BYTES_PER_SPAN / 2 ** 30:>16.2f} GB'
          f'{open_traces.mean():>20.3%}'
          f'{spans[open_traces].sum() / spans.sum():>23.1%}')
print()
print(f'slowest trace in the population: {duration_ms.max() / 1000:.0f} s')
print(f'a 30 s window truncates {(duration_ms > 30_000).sum():,} traces, of which '
      f'{is_error[duration_ms > 30_000].mean():.0%} ended in an error')

This is the constraint that decides whether tail sampling is available to you at all. The memory is
linear in your span rate and in the window, and the window has to be longer than your slowest trace,
which means the policy is most expensive exactly for the services whose traces are long and slow,
which are the services you most want it for.

The last two columns are the failure mode that follows. A trace still open when its window closes is
judged on the spans that arrived, so the sampler's view of a slow request is a truncated one, and the
spans that arrive afterwards are dropped as members of a trace that no longer exists. Those
percentages look small in trace counts and large in span counts, which is the signature of the
problem: the traces that outlast the window are the big ones, and most of them failed.

## Why the head decision is made once, at the root

In [ ]:
median_spans = int(np.median(spans))
multi = spans >= 5

print('if every service sampled independently at 1%:')
print(f'  a median trace of {median_spans} spans survives with probability '
      f'{HEAD_RATE ** median_spans:.0e}')
print(f'  expected intact traces of 5+ spans, out of {multi.sum():,}: '
      f'{(HEAD_RATE ** spans[multi].astype(float)).sum():.2e}')
print(f'  spans kept anyway: {HEAD_RATE * spans.sum():,.0f} '
      f'({HEAD_RATE:.0%} of the bandwidth, all of it orphan spans)')
print()

kept = policies['head 1%']
print('with one decision at the root, propagated down the call tree:')
print(f'  intact traces: {kept.sum():,}')
print(f'  spans kept: {spans[kept].sum():,} ({spans[kept].sum() / spans.sum():.2%} of all spans)')
print()

print('the same exponent during a rollout, when only some hops propagate context:')
for instrumented in [1.0, 0.95, 0.9, 0.7, 0.5]:
    survives = instrumented ** spans.astype(float)
    print(f'  {instrumented:>4.0%} of hops propagating -> '
          f'{survives.mean():>7.2%} of traces arrive complete, '
          f'{(survives > 0.5).mean():>6.1%} of traces are more likely intact than not')

The first block is the arithmetic behind a rule that sounds like dogma until you see the exponent: a
sampling decision belongs to a trace, not to a span. One percent raised to the power of a median
trace's span count is a number with dozens of zeros after the decimal point, so an independent
per-service decision produces no usable traces at all while still paying the full one percent of the
bandwidth. This is why the sampled flag travels in the propagation header: the root decides, and
every hop obeys.

The last block is the same exponent wearing a different hat, and it is why a tracing rollout feels
worthless until it is nearly finished. Even with nineteen hops in twenty propagating context, most
traces arrive broken, because a trace needs every hop to cooperate and the probability multiplies.
The useful unit of a tracing project is therefore a complete request path, instrumented end to end,
rather than a count of services with a library linked in.

## Sizing a policy against a question

In [ ]:
QUESTION_RATE = 0.0002          # the refund route, 1 trace in 5,000
WANTED = 100                    # examples needed to characterise a bug
ROUTE_PER_DAY = TRACE_RATE * 86_400 * QUESTION_RATE


def refund_retention(name, keep):
    """How often a refund trace survives the policy.

    Only the override treats the route specially, so for every other policy a refund
    trace is retained at the same rate as any other trace. Measuring that over all
    200,000 traces is far steadier than counting the 40-odd refunds in the sample.
    """
    return 1.0 if 'refunds always' in name else keep.mean()


print(f'the refund route runs {ROUTE_PER_DAY:,.0f} times a day at {TRACE_RATE:,} traces/s.')
print(f'to characterise a bug you want {WANTED} traced examples of it.')
print()
print(f'{"policy":>30}{"traces kept/day":>18}{"refunds kept/day":>19}{"time to 100":>14}')
for name, keep in list(policies.items()) + [('keep everything', np.ones(TRACES, bool))]:
    per_day = TRACE_RATE * 86_400 * keep.mean()
    examples = ROUTE_PER_DAY * refund_retention(name, keep)
    days = WANTED / examples
    eta = f'{days * 24 * 60:.0f} min' if days < 1 / 24 else (
        f'{days * 24:.1f} h' if days < 1 else f'{days:.1f} days')
    print(f'{name:>30}{per_day:>18,.0f}{examples:>19,.0f}{eta:>14}')
print()
print('the same question on a service a hundred times smaller (50 traces/s):')
for name, keep in policies.items():
    examples = 50 * 86_400 * QUESTION_RATE * refund_retention(name, keep)
    days = WANTED / examples
    print(f'{name:>30}{examples:>19.1f}'
          f'{(f"{days:.1f} days" if days < 365 else "never, in practice"):>14}')

This is the calculation worth doing before any sampling argument, because it turns a preference into
a number. A one percent head sample of a route that is two hundredths of a percent of traffic still
yields hundreds of examples a day at this volume, which is plenty. The second block is the same
policy on a service a hundred times smaller, where the same one percent yields nine examples a day
and the answer is a fortnight away.

That is why the third policy exists. A per-route override that always keeps refunds costs 2.3 percent
more data than head one percent, because the route is rare, and it answers in minutes what the global
policy answers in hours, or at the smaller scale in a fortnight.
Most of the value in a sampling configuration lives in a few overrides like it rather than in the
global rate, and the override is also the easiest thing to justify in review: here is the question,
here is how long it currently takes to answer, here is what the rule costs.

## What to take away

- Head sampling preserves the distribution and loses the rare events. Tail sampling preserves the
  rare events and destroys the distribution. Pick per question, and expect to run both.
- Never compute a percentile from a tail sampled population. Percentiles come from metrics, which
  see every request; traces explain the shape rather than measure it.
- Tail sampling costs memory proportional to span rate times decision window, and the window has to
  outlast your slowest trace. Price it before promising it.
- A sampling decision belongs to the trace and is made once, at the root. Independent per-span
  decisions keep bandwidth and produce nothing usable.
- Partial context propagation fails the same way, exponentially, so a tracing rollout delivers almost
  nothing until the last hops are instrumented.
- Size the policy against a question: how many examples of the thing you care about do you get per
  day. A targeted override usually beats raising the global rate, and it is cheaper to defend.